# PyG GraphXAI — 특허 소송 GNN 설명가능성

**`torch_geometric.explain`** = PyG 2.x에서 제공하는 공식 GNN XAI 프레임워크.  
standalone `graphxai` 라이브러리는 PyG 1.x용이며 현재 미유지. PyG 2.x에서는 이 프레임워크가 대체.

| 방법 | 이 노트북 | viz_explain_v2.ipynb (커스텀) |
|------|-----------|-------------------------------|
| GNNExplainer | `torch_geometric.explain.GNNExplainer` | `LinkGNNExplainer` 직접 구현 (M4) |
| PGExplainer  | `torch_geometric.explain.PGExplainer`  | `PGEdgeMLP` 직접 구현 (M5) |
| Attention    | GAT attention 수동 추출                | `compute_attention_flow` (M1) |

**이 노트북의 목표**  
1. PyG 공식 프레임워크로 GNN 예측 결과를 설명하는 방법 데모  
2. Citation 서브그래프(patent-patent 인용 관계)에서 소송 예측 근거 탐색  
3. 커스텀 구현 대비 장단점 비교

**실행 환경**: `patent_eval` conda 환경 (PyG 2.8.0, torch 2.6.0+cu124)

## 0. 임포트 및 환경 확인

In [ ]:
import sys, warnings, copy
sys.path.insert(0, 'viz_app')
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

# ── PyG XAI 프레임워크 ──────────────────────────────────
import torch_geometric
from torch_geometric.explain import Explainer, GNNExplainer, PGExplainer
from torch_geometric.nn import GATConv
from torch_geometric.utils import k_hop_subgraph

from backend.model import SimpleCatHeteroGAT, EDGE_TYPES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyG     : {torch_geometric.__version__}")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")

## 1. 모델 및 데이터 로드

In [ ]:
ckpt = torch.load('phase3_simplecat_ckpt.pt', map_location='cpu', weights_only=False)
data = torch.load('hetero_data.pt',            map_location='cpu', weights_only=False).to(device)

model = SimpleCatHeteroGAT(n_ipc=len(ckpt['ipc_list'])).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

pat_list    = ckpt['pat_list']
ipc_list    = ckpt['ipc_list']
co_list     = ckpt['co_list']
best_thr    = float(ckpt['best_threshold'])
sued_e      = np.array(ckpt['sued_edges'])      # (2, N_all)
tr_idx      = np.array(ckpt['train_edge_idx'])
te_idx      = np.array(ckpt['test_edge_idx'])
train_edges = sued_e[:, tr_idx]
test_edges  = sued_e[:, te_idx]
N_pat       = len(pat_list)

# 헤테로 모델 임베딩 (frozen)
with torch.no_grad():
    h_all = model(data)
h_pat = h_all['patent'].detach().cpu()   # (N_pat, 64)
h_np  = h_pat.numpy()

# Test pair 예측 확률
test_pairs = []
for j in range(test_edges.shape[1]):
    s, d = int(test_edges[0,j]), int(test_edges[1,j])
    prob = float(1 / (1 + np.exp(-float(np.dot(h_np[s], h_np[d])))))
    test_pairs.append((s, d, prob))
test_pairs.sort(key=lambda x: -x[2])

QUERY_S, QUERY_D, QUERY_P = test_pairs[0]
print(f"특허 수      : {N_pat:,}")
print(f"Train 소송   : {train_edges.shape[1]}개")
print(f"Test  소송   : {test_edges.shape[1]}개")
print()
print(f"[분석 대상]")
print(f"  원고 : {pat_list[QUERY_S]}  (idx={QUERY_S})")
print(f"  피고 : {pat_list[QUERY_D]}  (idx={QUERY_D})")
print(f"  P(sue) = {QUERY_P:.4f}  (threshold={best_thr:.2f})")

## 2. Patent Citation 서브그래프 구성

PyG의 `GNNExplainer` / `PGExplainer`는 단일 `(x, edge_index)` 인터페이스를 가정.  
헤테로 모델을 직접 넣을 수 없으므로 **patent-patent citation 동종 그래프**를 분리해 사용.

> ⚠️ **주의**: sued/sued_by 엣지는 제외 (데이터 누출 방지). cites + cited_by만 사용.

In [ ]:
# cites + cited_by → 무방향 인용 그래프
cites_ei    = data.edge_index_dict[('patent','cites','patent')].cpu()
cited_by_ei = data.edge_index_dict[('patent','cited_by','patent')].cpu()
pat_pat_ei  = torch.cat([cites_ei, cited_by_ei, 
                          cites_ei.flip(0), cited_by_ei.flip(0)], dim=1)
pat_pat_ei  = torch.unique(pat_pat_ei, dim=1)   # 중복 제거

print(f"Patent-patent 인용 엣지 : {pat_pat_ei.shape[1]:,}개")

# 2-hop 서브그래프 (원고·피고 중심)
sub_nodes, sub_ei, _, _ = k_hop_subgraph(
    node_idx=torch.tensor([QUERY_S, QUERY_D]),
    num_hops=2,
    edge_index=pat_pat_ei,
    relabel_nodes=True,
    num_nodes=N_pat,
)

# 서브그래프 내 원고·피고 인덱스
sub_node_list = sub_nodes.tolist()
src_sub = sub_node_list.index(QUERY_S)
dst_sub = sub_node_list.index(QUERY_D)

print(f"2-hop 서브그래프 노드 : {len(sub_nodes)}개")
print(f"2-hop 서브그래프 엣지 : {sub_ei.shape[1]}개")
print(f"원고 sub-idx={src_sub}, 피고 sub-idx={dst_sub}")

## 3. CitationGAT 정의 및 학습

**입력**: 헤테로 모델의 frozen 64d 임베딩 (특징으로 사용)  
**구조**: 2-layer GAT  
**학습 목표**: 특허 인용 관계 → 소송 예측 (BCE loss)  

GNNExplainer는 이 모델의 예측에 대해 **어떤 citation edge가 중요한지** 설명.

In [ ]:
class CitationGAT(nn.Module):
    """
    Frozen 64d hetero 임베딩 위에 얹은 2-layer GAT.

    forward(x, edge_index, edge_label_index=None)
      - edge_label_index 있으면: (E,) 소송 확률  → GNNExplainer (task_level='edge')
      - edge_label_index 없으면: (N, out) sigmoid 노드 임베딩  → PGExplainer (task_level='node')
    """
    def __init__(self, in_dim=64, hid=32, out_dim=32):
        super().__init__()
        self.conv1 = GATConv(in_dim, hid, heads=4, concat=True,
                             add_self_loops=False, dropout=0.1)
        self.conv2 = GATConv(hid*4, out_dim, heads=1, concat=False,
                             add_self_loops=False)

    def encode(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        return self.conv2(h, edge_index)        # (N, out_dim)

    def forward(self, x, edge_index, edge_label_index=None, **kw):
        h = self.encode(x, edge_index)
        if edge_label_index is not None:
            src_h = h[edge_label_index[0]]
            dst_h = h[edge_label_index[1]]
            return torch.sigmoid((src_h * dst_h).sum(dim=-1))  # (E,)
        # PGExplainer는 내부에서 BCE를 사용하므로 반드시 [0,1] 범위로 반환
        return torch.sigmoid(h)                 # (N, out_dim)


cite_gat = CitationGAT(in_dim=64, hid=32, out_dim=32).to(device)
print(f"파라미터 수: {sum(p.numel() for p in cite_gat.parameters()):,}")

In [ ]:
# ── 학습 데이터 준비 ──────────────────────────────────────
rng         = np.random.default_rng(42)
x_full      = h_pat.to(device)                           # (N_pat, 64) frozen
ei_full     = pat_pat_ei.to(device)                      # (2, E)

# 양성: train 소송 엣지
pos_s = torch.tensor(train_edges[0], dtype=torch.long, device=device)
pos_d = torch.tensor(train_edges[1], dtype=torch.long, device=device)

# 음성: 랜덤 비소송 쌍
neg_s = torch.tensor(rng.integers(0, N_pat, len(pos_s)), dtype=torch.long, device=device)
neg_d = torch.tensor(rng.integers(0, N_pat, len(pos_s)), dtype=torch.long, device=device)

eli_all = torch.stack([torch.cat([pos_s, neg_s]),
                        torch.cat([pos_d, neg_d])])       # (2, N_train*2)
y_all   = torch.cat([torch.ones(len(pos_s)),
                      torch.zeros(len(neg_s))]).to(device)

print(f"학습 엣지: {eli_all.shape[1]:,}개  (pos={len(pos_s)}, neg={len(neg_s)})")

# ── 학습 루프 ─────────────────────────────────────────────
optimizer = torch.optim.Adam(cite_gat.parameters(), lr=3e-3, weight_decay=1e-4)
cite_gat.train()

loss_hist = []
for epoch in range(300):
    optimizer.zero_grad()
    scores = cite_gat(x_full, ei_full, edge_label_index=eli_all)
    loss   = F.binary_cross_entropy(scores, y_all)
    loss.backward(); optimizer.step()
    if (epoch + 1) % 50 == 0:
        loss_hist.append(loss.item())
        print(f"  Epoch {epoch+1:3d} | Loss = {loss.item():.4f}")

cite_gat.eval()

# 분석 대상 쌍 예측 확인
with torch.no_grad():
    eli_q = torch.tensor([[QUERY_S],[QUERY_D]], device=device)
    p_cite = float(cite_gat(x_full, ei_full, edge_label_index=eli_q))

print(f"\nCitationGAT P(sue) = {p_cite:.4f}")
print(f"HeteroGAT   P(sue) = {QUERY_P:.4f}  ← 원본 모델")

## 4. PyG GNNExplainer

**핵심 질문**: *"원고-피고 특허 쌍의 소송 예측에 어떤 citation 연결이 중요한가?"*  

GNNExplainer는 `edge_index` 위에 soft mask $m_{ij} \in [0,1]$을 학습.  
mask를 적용했을 때 예측이 유지되면 → 그 엣지가 중요.

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.explain.config import ModelConfig

explainer_gnn = Explainer(
    model=cite_gat,
    algorithm=GNNExplainer(epochs=200, lr=0.01),
    explanation_type='model',
    node_mask_type='attributes',   # 64d 임베딩 차원별 중요도
    edge_mask_type='object',       # 엣지별 중요도
    model_config=ModelConfig(
        mode='binary_classification',
        task_level='edge',
        return_type='probs',
    ),
)

print("GNNExplainer 학습 중 (약 5~15초)...")
eli_q = torch.tensor([[QUERY_S],[QUERY_D]], device=device)

explanation_gnn = explainer_gnn(
    x=x_full,
    edge_index=ei_full,
    edge_label_index=eli_q,
)

print(f"완료")
print(f"  node_mask : {explanation_gnn.node_mask.shape}")
print(f"  edge_mask : {explanation_gnn.edge_mask.shape}")

em  = explanation_gnn.edge_mask.cpu().detach().numpy()
nm  = explanation_gnn.node_mask.cpu().detach().numpy()  # (N_pat, 64)
ei_np = ei_full.cpu().numpy()

# 상위 edge mask
top10_idx = np.argsort(-em)[:10]
print(f"\n상위 10개 citation edge mask:")
for rank, i in enumerate(top10_idx):
    s, d = int(ei_np[0,i]), int(ei_np[1,i])
    print(f"  {rank+1:2d}. {str(pat_list[s])[:18]:18s} → {str(pat_list[d])[:18]:18s}  mask={em[i]:.4f}")

In [ ]:
# ── GNNExplainer 시각화 ──────────────────────────────────
fig = plt.figure(figsize=(18, 7), facecolor='#0e1117')
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
fig.suptitle(
    f'PyG GNNExplainer | {pat_list[QUERY_S]} → {pat_list[QUERY_D]}  P={QUERY_P:.3f}',
    color='white', fontsize=12, y=1.01)

# --- (A) Node feature importance (64 dims of frozen embedding) ---
ax_feat = fig.add_subplot(gs[0])
ax_feat.set_facecolor('#1e2130')
feat_imp = np.abs(nm).mean(axis=0)  # mean over N_pat nodes → (64,)
top20_f  = np.argsort(-feat_imp)[:20]
colors_f = ['#4fc3f7' if v > feat_imp.mean() else '#78909c' for v in feat_imp[top20_f]]
ax_feat.bar(range(20), feat_imp[top20_f], color=colors_f, alpha=0.85)
ax_feat.set_title('임베딩 피처 중요도 (Top-20 dims)', color='white', fontsize=10)
ax_feat.set_xlabel('Embedding dimension', color='gray', fontsize=8)
ax_feat.set_ylabel('|node_mask|', color='gray', fontsize=8)
ax_feat.tick_params(colors='white', labelsize=7)
for s in ax_feat.spines.values(): s.set_edgecolor('#444')

# --- (B) Edge mask distribution ---
ax_hist = fig.add_subplot(gs[1])
ax_hist.set_facecolor('#1e2130')
ax_hist.hist(em, bins=40, color='#4fc3f7', alpha=0.8, edgecolor='none')
ax_hist.axvline(0.5, color='yellow', linestyle='--', linewidth=1.5, label='threshold=0.5')
n_important = int((em > 0.5).sum())
ax_hist.set_title(f'Edge Mask 분포  (>0.5: {n_important}개)', color='white', fontsize=10)
ax_hist.set_xlabel('mask value', color='gray', fontsize=8)
ax_hist.set_ylabel('# edges', color='gray', fontsize=8)
ax_hist.tick_params(colors='white', labelsize=7)
ax_hist.legend(fontsize=7, facecolor='#1e2130', labelcolor='white')
for s in ax_hist.spines.values(): s.set_edgecolor('#444')

# --- (C) Top-K important edges ---
ax_top = fig.add_subplot(gs[2])
ax_top.set_facecolor('#1e2130')
K = 15
top_k_idx = np.argsort(-em)[:K]
edge_lbls  = []
for i in top_k_idx:
    s, d = int(ei_np[0,i]), int(ei_np[1,i])
    sl = str(pat_list[s])[:12]
    dl = str(pat_list[d])[:12]
    edge_lbls.append(f"{sl}→{dl}")
bar_colors = ['#ef5350' if v > 0.5 else '#42a5f5' for v in em[top_k_idx]]
ax_top.barh(range(K), em[top_k_idx], color=bar_colors, alpha=0.85)
ax_top.axvline(0.5, color='yellow', linestyle='--', linewidth=1.2, alpha=0.7)
ax_top.set_yticks(range(K))
ax_top.set_yticklabels(edge_lbls, fontsize=6, color='white')
ax_top.set_xlabel('edge mask value', color='gray', fontsize=8)
ax_top.set_title(f'중요 Citation 엣지 Top-{K}', color='white', fontsize=10)
ax_top.tick_params(colors='white', labelsize=7)
for s in ax_top.spines.values(): s.set_edgecolor('#444')

# 빨강=중요(>0.5), 파랑=덜중요
from matplotlib.patches import Patch
ax_top.legend(handles=[
    Patch(color='#ef5350', label='mask>0.5 (중요)'),
    Patch(color='#42a5f5', label='mask≤0.5'),
], fontsize=6, facecolor='#1e2130', labelcolor='white')

plt.tight_layout()
plt.savefig('graphxai_gnnexplainer.png', dpi=130, bbox_inches='tight')
plt.show()
print("저장: graphxai_gnnexplainer.png")

## 5. PyG PGExplainer

**GNNExplainer 한계**: 각 쌍마다 독립적으로 학습 → 다른 쌍에 재사용 불가.  
**PGExplainer**: 전체 그래프에 공통된 MLP $\phi(h_i, h_j)$를 학습 → 한 번 학습 후 모든 쌍에 적용.

여기서는 **노드 임베딩 재현**을 목표로 학습:  
"원고/피고 각각의 임베딩 형성에 어떤 citation 연결이 기여했는가?"

In [ ]:
from torch_geometric.explain import Explainer, PGExplainer

# PyG 2.8.0 이슈: PGExplainer 내부 MLP가 CPU에서 초기화될 수 있음
# → 안전하게 CPU 텐서로 사용 (N_pat=1331, E=1606 → CPU도 충분히 빠름)
cite_gat_cpu = copy.deepcopy(cite_gat).cpu()
x_cpu        = x_full.cpu()
ei_cpu       = ei_full.cpu()

explainer_pg = Explainer(
    model=cite_gat_cpu,
    algorithm=PGExplainer(epochs=30, lr=3e-3),
    explanation_type='phenomenon',
    edge_mask_type='object',
    model_config=ModelConfig(
        mode='binary_classification',
        task_level='node',
        return_type='probs',
    ),
)

with torch.no_grad():
    target_cpu = cite_gat_cpu(x_cpu, ei_cpu)   # (N_pat, 32), sigmoid → [0,1]

train_nodes = sorted(set(train_edges[0].tolist() + train_edges[1].tolist()))[:80]
print(f"PGExplainer 학습 노드 수: {len(train_nodes)}")
print(f"target 범위: [{target_cpu.min():.3f}, {target_cpu.max():.3f}]")
print("학습 중 (약 10~30초)...")

for epoch in range(1, 31):
    for node_idx in train_nodes:
        if node_idx >= N_pat:
            continue
        explainer_pg.algorithm.train(
            epoch=epoch,
            model=cite_gat_cpu,
            x=x_cpu,
            edge_index=ei_cpu,
            target=target_cpu,
            index=node_idx,
        )
    if epoch % 10 == 0:
        print(f"  Epoch {epoch}/30")

print("학습 완료")

In [ ]:
# ── 원고 / 피고 각각 설명 생성 (CPU) ────────────────────────
exp_pg_src = explainer_pg(
    x=x_cpu, edge_index=ei_cpu,
    target=target_cpu, index=QUERY_S,
)
exp_pg_dst = explainer_pg(
    x=x_cpu, edge_index=ei_cpu,
    target=target_cpu, index=QUERY_D,
)

em_pg_src = exp_pg_src.edge_mask.cpu().detach().numpy()
em_pg_dst = exp_pg_dst.edge_mask.cpu().detach().numpy()
ei_np     = ei_cpu.numpy()    # CPU edge_index numpy

print(f"원고 PGExplainer edge_mask: {em_pg_src.shape}")
print(f"피고 PGExplainer edge_mask: {em_pg_dst.shape}")

def topk_edges(em, ei_np, pat_list, k=10, node_focus=None):
    """node_focus 주변 엣지에 가중치 부여 후 상위 k 반환."""
    if node_focus is not None:
        focus_mask = (ei_np[0] == node_focus) | (ei_np[1] == node_focus)
        scores = em.copy()
        scores[~focus_mask] *= 0.5
    else:
        scores = em
    top_idx = np.argsort(-scores)[:k]
    return [(str(pat_list[int(ei_np[0,i])])[:14],
             str(pat_list[int(ei_np[1,i])])[:14],
             float(em[i])) for i in top_idx]

print(f"\n[원고 {pat_list[QUERY_S][:20]} 중심 중요 citation]")
for sl, dl, v in topk_edges(em_pg_src, ei_np, pat_list, k=8, node_focus=QUERY_S):
    print(f"  {sl:14s} → {dl:14s}  score={v:.4f}")

print(f"\n[피고 {pat_list[QUERY_D][:20]} 중심 중요 citation]")
for sl, dl, v in topk_edges(em_pg_dst, ei_np, pat_list, k=8, node_focus=QUERY_D):
    print(f"  {sl:14s} → {dl:14s}  score={v:.4f}")

In [ ]:
# ── PGExplainer 시각화 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0e1117')
fig.suptitle(
    f'PyG PGExplainer — 원고·피고 임베딩 형성에 기여한 Citation 엣지\n'
    f'({pat_list[QUERY_S]} → {pat_list[QUERY_D]})',
    color='white', fontsize=11, y=1.02)

def plot_pg_edges(ax, em, ei_np, pat_list, node_focus, title):
    ax.set_facecolor('#1e2130')
    focus_mask = (ei_np[0] == node_focus) | (ei_np[1] == node_focus)
    em_focus   = em.copy(); em_focus[~focus_mask] = 0
    top_idx    = np.argsort(-em_focus)[:12]
    lbls = []
    for i in top_idx:
        s, d = int(ei_np[0,i]), int(ei_np[1,i])
        sl   = str(pat_list[s])[:13]
        dl   = str(pat_list[d])[:13]
        marker = '★' if s==node_focus or d==node_focus else '·'
        lbls.append(f"{marker} {sl}→{dl}")
    vals   = em_focus[top_idx]
    colors = ['#ef5350' if v > np.median(vals[vals>0]) else '#42a5f5' for v in vals]
    ax.barh(range(len(top_idx)), vals, color=colors, alpha=0.85)
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels(lbls, fontsize=7, color='white')
    ax.set_xlabel('PGExplainer edge score', color='gray', fontsize=8)
    ax.set_title(title, color='white', fontsize=10)
    ax.tick_params(colors='white', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#444')

plot_pg_edges(axes[0], em_pg_src, ei_np, pat_list, QUERY_S,
              f'원고 {str(pat_list[QUERY_S])[:18]} 임베딩 기여 엣지')
plot_pg_edges(axes[1], em_pg_dst, ei_np, pat_list, QUERY_D,
              f'피고 {str(pat_list[QUERY_D])[:18]} 임베딩 기여 엣지')

plt.tight_layout()
plt.savefig('graphxai_pgexplainer.png', dpi=130, bbox_inches='tight')
plt.show()
print("저장: graphxai_pgexplainer.png")

## 6. GNNExplainer vs PGExplainer — 서브그래프 겹침 분석

두 방법이 "중요하다"고 판단한 citation edge가 얼마나 겹치는지 확인.

In [ ]:
THRESH_GNN = 0.5
THRESH_PG  = float(np.percentile(em_pg_src, 85))  # 상위 15%

def edge_set(em, ei_np, threshold):
    idx = np.where(em > threshold)[0]
    return {(int(ei_np[0,i]), int(ei_np[1,i])) for i in idx}

# GNNExplainer edge_mask는 ei_full (CUDA→CPU) 기준
ei_np_gnn = ei_full.cpu().numpy()
set_gnn = edge_set(em,        ei_np_gnn, THRESH_GNN)
set_pgs = edge_set(em_pg_src, ei_np,     THRESH_PG)
set_pgd = edge_set(em_pg_dst, ei_np,     THRESH_PG)

overlap_gnn_pgs = set_gnn & set_pgs
overlap_gnn_pgd = set_gnn & set_pgd
overlap_both_pg = set_pgs & set_pgd

print(f"GNNExplainer 중요 엣지 (>{THRESH_GNN:.2f})   : {len(set_gnn):4d}개")
print(f"PGExplainer[원고] 중요 엣지 (>{THRESH_PG:.2f}) : {len(set_pgs):4d}개")
print(f"PGExplainer[피고] 중요 엣지 (>{THRESH_PG:.2f}) : {len(set_pgd):4d}개")
print()
print(f"GNN ∩ PG[원고]  : {len(overlap_gnn_pgs):3d}개")
print(f"GNN ∩ PG[피고]  : {len(overlap_gnn_pgd):3d}개")
print(f"PG[원고] ∩ PG[피고] (Bridge): {len(overlap_both_pg):3d}개")

if overlap_both_pg:
    print("\n[Bridge citations — 원고·피고 양쪽에 중요한 공통 인용 관계]")
    for s, d in list(overlap_both_pg)[:5]:
        print(f"  {str(pat_list[s])[:20]:20s} → {str(pat_list[d])[:20]:20s}")

## 7. Attention 직접 추출 (HeteroGAT 원본 모델)

PyG의 `AttentionExplainer`는 `HeteroConv` + heterogeneous input을 직접 지원하지 않음.  
대신 GAT attention weights를 수동으로 추출해 동일한 정보를 얻는다.  
(이게 M1이 하는 일과 동일하며, 원본 HeteroGAT를 그대로 사용하는 유일한 방법)

In [ ]:
def extract_attn_flow(model, data, target_idx, top_n=15):
    """
    PyG AttentionExplainer의 핵심 로직을 HeteroGAT에 직접 구현.
    레이어별로 attention weight를 수집 후 역전파 방식으로 집계.
    
    Returns: {hop: [(node_idx, accumulated_flow), ...]}
    """
    x_pat_raw = torch.cat([data['patent'].struct, data['patent'].x], dim=1)
    x = {
        'patent':  model.pat_proj(x_pat_raw),
        'company': F.leaky_relu(model.co_proj(data['company'].x)),
        'ipc':     model.ipc_emb.weight.clone(),
    }
    acts = [F.leaky_relu, F.leaky_relu, torch.tanh]
    layer_d2s = []   # layer i → {dst: [(src, attn)]}

    for li, (hconv, bn, act) in enumerate(zip(model.convs[:3], model.bns, acts)):
        x_agg = defaultdict(list)
        d2s   = defaultdict(list)
        for et in EDGE_TYPES:
            src_t, rel, dst_t = et
            if et not in data.edge_index_dict or et not in hconv.convs:
                continue
            et_conv  = hconv.convs[et]
            edge_idx = data.edge_index_dict[et]
            with torch.no_grad():
                out, (ei_ret, alpha) = et_conv(
                    (x[src_t], x[dst_t]), edge_idx,
                    return_attention_weights=True)
            if src_t == 'patent' and dst_t == 'patent':
                am   = alpha.mean(dim=1).cpu().numpy()
                srcs = ei_ret[0].cpu().numpy()
                dsts = ei_ret[1].cpu().numpy()
                for s, d, a in zip(srcs, dsts, am):
                    d2s[int(d)].append((int(s), float(a)))
            x_agg[dst_t].append(out)
        layer_d2s.append(dict(d2s))
        x_new = {k: torch.stack(vs).mean(0) for k, vs in x_agg.items() if vs}
        x = {k: (model.drop(act(bn[k](v.view(-1, model.hid)))) if k in bn
                 else model.drop(act(v))) for k, v in x_new.items()}

    # 역추적: L3→L2→L1, hop 1→2→3
    current = {target_idx: 1.0}
    result  = {}
    for li in range(2, -1, -1):
        new_flow = {}
        for dst_n, dst_w in current.items():
            for src_n, a_w in layer_d2s[li].get(dst_n, []):
                if src_n == target_idx:
                    continue
                new_flow[src_n] = new_flow.get(src_n, 0.0) + dst_w * a_w
        hop = 3 - li
        result[hop] = sorted(new_flow.items(), key=lambda x: -x[1])[:top_n]
        current = new_flow
    return result


print("Attention flow 추출 중...")
flow_src = extract_attn_flow(model, data, QUERY_S, top_n=15)
flow_dst = extract_attn_flow(model, data, QUERY_D, top_n=15)
print("완료")

for node_lbl, flow in [(f'원고 {pat_list[QUERY_S][:15]}', flow_src),
                        (f'피고 {pat_list[QUERY_D][:15]}', flow_dst)]:
    print(f"\n[{node_lbl}]")
    for hop, items in flow.items():
        top3 = items[:3]
        print(f"  hop-{hop}: " + ", ".join(f"{str(pat_list[n])[:12]}({v:.3f})"
                                            for n, v in top3))

In [ ]:
# ── Attention Flow 시각화 ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#0e1117')
fig.suptitle(
    f'Attention Flow (HeteroGAT 원본 모델)\n'
    f'{pat_list[QUERY_S]} → {pat_list[QUERY_D]}',
    color='white', fontsize=12)

hop_colors = {1: '#ef5350', 2: '#ff9800', 3: '#ffeb3b'}

for row, (flow, node_lbl) in enumerate([
    (flow_src, f'원고 {str(pat_list[QUERY_S])[:20]}'),
    (flow_dst, f'피고 {str(pat_list[QUERY_D])[:20]}'),
]):
    for col, hop in enumerate([1, 2, 3]):
        ax = axes[row][col]
        ax.set_facecolor('#1e2130')
        items = flow.get(hop, [])
        if not items:
            ax.text(0.5, 0.5, '데이터 없음', ha='center', color='gray')
            ax.axis('off')
            continue
        lbls = [str(pat_list[n])[:16] for n, _ in items[:10]]
        vals = [v for _, v in items[:10]]
        ax.barh(range(len(lbls)), vals, color=hop_colors[hop], alpha=0.8)
        ax.set_yticks(range(len(lbls)))
        ax.set_yticklabels(lbls, fontsize=6, color='white')
        ax.set_xlabel('Attention flow', color='gray', fontsize=7)
        ax.set_title(f'{node_lbl[:22]} | {hop}-hop', color='white', fontsize=8)
        ax.tick_params(colors='white', labelsize=6)
        for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig('graphxai_attention_flow.png', dpi=130, bbox_inches='tight')
plt.show()
print("저장: graphxai_attention_flow.png")

## 8. 종합 비교

### PyG `torch_geometric.explain` vs 커스텀 M1–M7

| 항목 | PyG GNNExplainer | PyG PGExplainer | Attention Flow (수동) | 커스텀 M4 | 커스텀 M5 |
|------|-----------------|-----------------|----------------------|-----------|----------|
| 설명 대상 | citation edge mask | citation edge score | GAT attention weight | sued edge mask | 이웃 patent score |
| 원본 모델 사용 | ✗ (surrogate) | ✗ (surrogate) | ✓ (직접) | ✓ (직접) | ✓ (직접) |
| 재사용성 | ✗ (쌍마다) | ✓ (학습 후 공용) | ✗ (쌍마다) | ✗ (쌍마다) | ✓ (MLP 공용) |
| 헤테로 지원 | ✗ (homo 래퍼 필요) | ✗ (homo 래퍼 필요) | ✓ (모든 엣지타입) | ✓ (엣지타입별) | ✓ (patent-patent) |
| 속도 | 중간 (~10s) | 느림 (학습 후 빠름) | 빠름 | 느림 (~30s) | 느림 (~20s) |
| 구현 난이도 | 쉬움 (API) | 중간 (학습 루프) | 중간 (수동) | 높음 | 높음 |

### 결론

1. **PyG 프레임워크의 장점**: 표준 API, 코드 간결, 공식 지원
2. **PyG 프레임워크의 한계**: HeteroConv + Link Prediction 조합에서 surrogate 모델이 필요 → **설명이 원본 모델이 아닌 surrogate를 설명**
3. **커스텀 구현의 장점**: 원본 HeteroGAT를 직접 설명, 엣지타입별 중요도 분리 가능
4. **권장**: 원본 모델 충실도가 중요하면 커스텀 M1–M7 유지. 팀 공유/프레임워크 통일이 목적이면 PyG 프레임워크 도입 가능 (단, surrogate 모델의 정확도 검증 필요)

In [ ]:
# ── 세 방법 비교 대시보드 ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7), facecolor='#0e1117')
fig.suptitle(
    f'XAI 방법 비교: {pat_list[QUERY_S]} → {pat_list[QUERY_D]}\n'
    f'(GNNExplainer · PGExplainer · Attention Flow)',
    color='white', fontsize=11)

# --- (A) GNNExplainer: edge mask 상위 k ---
ax = axes[0]
ax.set_facecolor('#1e2130')
K = 12
top_idx_gnn = np.argsort(-em)[:K]
lbls_a = []
for i in top_idx_gnn:
    s, d = int(ei_np[0,i]), int(ei_np[1,i])
    lbls_a.append(f"{str(pat_list[s])[:11]}→{str(pat_list[d])[:11]}")
ax.barh(range(K), em[top_idx_gnn], color='#4fc3f7', alpha=0.85)
ax.axvline(0.5, color='yellow', ls='--', lw=1.2, alpha=0.7)
ax.set_yticks(range(K)); ax.set_yticklabels(lbls_a, fontsize=6, color='white')
ax.set_title('GNNExplainer\n(Citation Edge Mask)', color='white', fontsize=10)
ax.set_xlabel('mask value', color='gray', fontsize=8)
ax.tick_params(colors='white'); [s.set_edgecolor('#444') for s in ax.spines.values()]

# --- (B) PGExplainer: 원고 중심 ---
ax = axes[1]
ax.set_facecolor('#1e2130')
focus_mask = (ei_np[0] == QUERY_S) | (ei_np[1] == QUERY_S)
em_focus   = em_pg_src.copy(); em_focus[~focus_mask] = 0
top_idx_pg = np.argsort(-em_focus)[:K]
lbls_b = []
for i in top_idx_pg:
    s, d = int(ei_np[0,i]), int(ei_np[1,i])
    lbls_b.append(f"{str(pat_list[s])[:11]}→{str(pat_list[d])[:11]}")
ax.barh(range(K), em_focus[top_idx_pg], color='#ab47bc', alpha=0.85)
ax.set_yticks(range(K)); ax.set_yticklabels(lbls_b, fontsize=6, color='white')
ax.set_title('PGExplainer [원고]\n(Parametric Edge Score)', color='white', fontsize=10)
ax.set_xlabel('edge score', color='gray', fontsize=8)
ax.tick_params(colors='white'); [s.set_edgecolor('#444') for s in ax.spines.values()]

# --- (C) Attention Flow: hop별 집계 ---
ax = axes[2]
ax.set_facecolor('#1e2130')
attn_nodes, attn_vals, attn_colors = [], [], []
seen = set()
for hop in [1, 2, 3]:
    for n, v in flow_src.get(hop, [])[:5]:
        if n not in seen:
            seen.add(n)
            attn_nodes.append(str(pat_list[n])[:16])
            attn_vals.append(v)
            attn_colors.append(hop_colors[hop])
ax.barh(range(len(attn_nodes)), attn_vals, color=attn_colors, alpha=0.85)
ax.set_yticks(range(len(attn_nodes)))
ax.set_yticklabels(attn_nodes, fontsize=6, color='white')
ax.set_title('Attention Flow [원고]\n(HeteroGAT 직접 추출)', color='white', fontsize=10)
ax.set_xlabel('accumulated attention', color='gray', fontsize=8)
ax.tick_params(colors='white'); [s.set_edgecolor('#444') for s in ax.spines.values()]
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=hop_colors[h], label=f'{h}-hop') for h in [1,2,3]],
          fontsize=7, facecolor='#1e2130', labelcolor='white')

plt.tight_layout()
plt.savefig('graphxai_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print("저장: graphxai_comparison.png")

In [ ]:
# ── 저장 파일 요약 ────────────────────────────────────────────
print("="*60)
print(f"분석 대상: {pat_list[QUERY_S]} → {pat_list[QUERY_D]}")
print(f"HeteroGAT P(sue): {QUERY_P:.4f}  |  CitationGAT P(sue): {p_cite:.4f}")
print("="*60)
outputs = [
    ('GNNExplainer', 'graphxai_gnnexplainer.png',
     f'중요 citation edge {int((em>0.5).sum())}개 탐색'),
    ('PGExplainer',  'graphxai_pgexplainer.png',
     '원고·피고 임베딩 형성 기여 엣지'),
    ('Attention Flow','graphxai_attention_flow.png',
     'HeteroGAT 원본 모델 attention (3-hop)'),
    ('비교 대시보드',  'graphxai_comparison.png',
     '세 방법 나란히 비교'),
]
for name, fname, desc in outputs:
    print(f"  [{name}] {fname}")
    print(f"    → {desc}")